# Week 2 — Logistics Data Collection, Cleaning & Preprocessing
## Nitesh Chauhan — Logistics Data Analyst Intern
This notebook demonstrates a reproducible preprocessing pipeline using a simulated 1,000-row logistics dataset modeled on the public DataCo SMART Supply Chain dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('Week2_Logistics_Raw_Simulated.csv')
print(df.shape)
display(df.head())

In [ ]:
# Initial profiling
print(df.info())
display(df.describe(include='all').T)
display(df.isna().sum().sort_values(ascending=False))
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Standardize column names and data types
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(r'[^a-z0-9_]', '', regex=True))
for col in ['order_date','shipping_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
for col in ['days_for_shipping_real','days_for_shipment_scheduled','order_item_quantity','sales','order_profit_per_order']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Clean categories and duplicates
df['shipping_mode'] = df['shipping_mode'].astype('string').str.strip().str.title()
print(df['shipping_mode'].value_counts(dropna=False))
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicates after:', df.duplicated().sum())

In [ ]:
# Convert invalid values to missing, then impute
df.loc[df['days_for_shipping_real'] < 0, 'days_for_shipping_real'] = np.nan
df.loc[df['sales'] < 0, 'sales'] = np.nan
for col in ['order_item_quantity','order_profit_per_order','days_for_shipping_real','sales']:
    df[col] = df[col].fillna(df[col].median())
      

In [ ]:
# IQR outlier detection
q1 = df['days_for_shipping_real'].quantile(.25)
q3 = df['days_for_shipping_real'].quantile(.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
df['shipping_time_outlier'] = ((df['days_for_shipping_real'] < lower) | (df['days_for_shipping_real'] > upper)).astype(int)
print('Outliers:', df['shipping_time_outlier'].sum())

In [ ]:
# Logistics feature engineering
df['delay_days'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']
df['sla_breach'] = (df['delay_days'] > 0).astype(int)
df['profit_margin_pct'] = (df['order_profit_per_order'] / df['sales']) * 100

In [ ]:
# Min-Max normalization
scale_cols = ['days_for_shipping_real','order_item_quantity','sales']
scaler = MinMaxScaler()
df[[c + '_normalized' for c in scale_cols]] = scaler.fit_transform(df[scale_cols])
display(df[scale_cols + [c + '_normalized' for c in scale_cols]].head())

In [ ]:
# Final validation and export
print('Rows:', len(df))
print('Duplicates:', df.duplicated().sum())
print('Remaining missing values:', int(df.isna().sum().sum()))
print('Negative shipping days:', int((df['days_for_shipping_real'] < 0).sum()))
df.to_csv('Week2_Logistics_Cleaned.csv', index=False)